#### Imports

In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

np.random.seed(42)
random.seed(42)

df_suppliers = pd.read_csv("../data/supplier_master.csv")
print(f"Suppliers loaded: {len(df_suppliers)}")

Suppliers loaded: 120


#### Config

In [2]:
N_EVENTS   = 8000
START_DATE = datetime(2024, 1, 1)
END_DATE   = datetime(2025, 3, 31)

VIOLATION_TYPES = [
    "ASN_LATE", "ASN_MISSING", "PKG_DEVIATION",
    "QTY_MISMATCH", "OTIF_MISS", "LABEL_ERROR",
    "PO_VIOLATION", "SHORTSHIP"
]

# Flat fee range (min, max) in USD
# Tuple of (min, max) — for % types, this is % of PO value
PENALTY_RULES = {
    "ASN_LATE":      ("flat",    150,    800),
    "ASN_MISSING":   ("flat",    400,   2000),
    "PKG_DEVIATION": ("flat",    200,   1500),
    "QTY_MISMATCH":  ("pct",   0.03,   0.08),
    "OTIF_MISS":     ("pct",   0.02,   0.05),
    "LABEL_ERROR":   ("flat",   100,    600),
    "PO_VIOLATION":  ("flat",   300,   2500),
    "SHORTSHIP":     ("pct",   0.05,   0.12),
}

# How likely each risk profile is to generate a violation per shipment
VIOLATION_PROB = {
    "Low":    0.06,
    "Medium": 0.15,
    "High":   0.30,
}

print("Config ready.")

Config ready.


#### Helper functions

In [3]:
def random_date(start, end):
    # Pick a random number of days between start and end
    delta = end - start
    return start + timedelta(days=random.randint(0, delta.days))

def calculate_penalty(violation_type, po_value):
    rule = PENALTY_RULES[violation_type]
    if rule[0] == "flat":
        return round(random.uniform(rule[1], rule[2]), 2)
    else:
        # pct = percentage of PO value
        return round(po_value * random.uniform(rule[1], rule[2]), 2)

print("Functions ready.")

Functions ready.


#### Generate events

In [4]:
events = []

for _ in range(N_EVENTS):
    # Pick a random supplier for this shipment
    supplier = df_suppliers.sample(1).iloc[0]

    ship_date = random_date(START_DATE, END_DATE)
    po_value  = random.randint(8_000, 450_000)
    qty_ordered = random.randint(50, 5000)

    # Does this shipment have a violation?
    violation_prob = VIOLATION_PROB[supplier["risk_profile"]]
    has_violation  = random.random() < violation_prob

    violation_type = None
    penalty_usd    = 0.0

    if has_violation:
        violation_type = random.choice(VIOLATION_TYPES)
        penalty_usd    = calculate_penalty(violation_type, po_value)

    events.append({
        "shipment_id":    f"SHP-{random.randint(100000, 999999)}",
        "supplier_id":    supplier["supplier_id"],
        "supplier_name":  supplier["supplier_name"],
        "region":         supplier["region"],
        "primary_plant":  supplier["primary_plant"],
        "tier":           supplier["tier"],
        "part_category":  supplier["part_category"],
        "risk_profile":   supplier["risk_profile"],
        "ship_date":      ship_date.strftime("%Y-%m-%d"),
        "ship_month":     ship_date.strftime("%Y-%m"),
        "ship_quarter":   f"Q{(ship_date.month-1)//3+1} {ship_date.year}",
        "po_value_usd":   po_value,
        "qty_ordered":    qty_ordered,
        "has_violation":  has_violation,
        "violation_type": violation_type if has_violation else "NONE",
        "penalty_usd":    penalty_usd,
    })

df_events = pd.DataFrame(events)
print(df_events.shape)
print(df_events.head())

(8000, 16)
  shipment_id supplier_id supplier_name       region  primary_plant    tier  \
0  SHP-356787    SUPP_045  Supplier_045   APAC-China  Giga Shanghai  Tier 1   
1  SHP-671858    SUPP_089  Supplier_089  EMEA-Europe    Giga Berlin  Tier 1   
2  SHP-629903    SUPP_084  Supplier_084      NA-West   Giga Fremont  Tier 1   
3  SHP-781453    SUPP_072  Supplier_072   APAC-China  Giga Shanghai  Tier 1   
4  SHP-717889    SUPP_066  Supplier_066      NA-East  Giga New York  Tier 3   

     part_category risk_profile   ship_date ship_month ship_quarter  \
0         Interior          Low  2024-11-23    2024-11      Q4 2024   
1         Interior       Medium  2024-04-24    2024-04      Q2 2024   
2        Fasteners          Low  2024-02-14    2024-02      Q1 2024   
3  Chassis & Frame          Low  2024-11-04    2024-11      Q4 2024   
4      Electronics       Medium  2024-12-25    2024-12      Q4 2024   

   po_value_usd  qty_ordered  has_violation violation_type  penalty_usd  
0         663

#### Sanity check

In [5]:
print("=== VIOLATION RATE ===")
print(df_events["has_violation"].value_counts())

print("\n=== VIOLATION TYPES ===")
print(df_events[df_events["has_violation"]]["violation_type"].value_counts())

print("\n=== PENALTY STATS ===")
print(df_events["penalty_usd"].describe().round(2))

print(f"\nTotal penalty value: ${df_events['penalty_usd'].sum():,.0f}")
print(f"Total chargebacks:   {df_events['has_violation'].sum():,}")

=== VIOLATION RATE ===
has_violation
False    6841
True     1159
Name: count, dtype: int64

=== VIOLATION TYPES ===
violation_type
PO_VIOLATION     155
ASN_MISSING      154
ASN_LATE         150
LABEL_ERROR      145
SHORTSHIP        144
QTY_MISMATCH     141
OTIF_MISS        138
PKG_DEVIATION    132
Name: count, dtype: int64

=== PENALTY STATS ===
count     8000.00
mean       770.55
std       3731.70
min          0.00
25%          0.00
50%          0.00
75%          0.00
max      48437.13
Name: penalty_usd, dtype: float64

Total penalty value: $6,164,426
Total chargebacks:   1,159


#### Save

In [6]:
df_events.to_csv("../data/shipment_events.csv", index=False)
print(f"Saved: {df_events.shape}")

Saved: (8000, 16)
